# Paris Data Preprocessing Pipeline
**Goal:** Create a master homogeneous dataframe for Paris (Quartier level) 2015-2025. 

**Data Sources:** 
* DVF+ (Real Estate Transactions)
* INSEE (Income, Population, Education)
* Paris Open Data (Geometry)

In [1]:
import os
import pandas as pd
import numpy as np
import geopandas as gpd
from pathlib import Path
from sklearn.impute import KNNImputer # for spatial interpolation (3-4 neighborhoods out of 80 have some missing features)

## Config

In [2]:
BASE_DIR = Path("../data/raw/paris")
OUTPUT_DIR = Path("../data/preprocessed/paris")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Geometry Path
GEO_PATH = BASE_DIR / "quartier_paris.geojson"

# 2. DVF Path
dvf_candidate_1 = BASE_DIR / "household prices/DVF_PLUS_2025_2_CSV_R011_ED251/1_DONNEES_LIVRAISON/dvf_plus_d75.csv"
dvf_candidate_2 = BASE_DIR / "household prices/dvf_plus_d75.csv"
DVF_PATH_PLUS = dvf_candidate_1 if dvf_candidate_1.exists() else dvf_candidate_2

# 3. Reference Table Path
REF_PATH = BASE_DIR / "IRIS reference table/Emboitements_IRIS.csv"

# Target Years
YEARS_POP = [2015, 2019, 2021, 2022]
YEARS_EDU = [2015, 2019, 2021, 2022]
YEARS_INC = [2015, 2019, 2021] 

print("=== CONFIGURATION ===")
print(f"Geometry: {GEO_PATH}")
print(f"DVF+:     {DVF_PATH_PLUS}")

=== CONFIGURATION ===
Geometry: ../data/raw/paris/quartier_paris.geojson
DVF+:     ../data/raw/paris/household prices/DVF_PLUS_2025_2_CSV_R011_ED251/1_DONNEES_LIVRAISON/dvf_plus_d75.csv


## 1. Geometry & ID Standardization (The Foundation)
We load the geometry ONCE and establish the "Global ID" format (`PARIS_751` + Arrond + Quartier) here.
All subsequent dataframes must map to these IDs.

In [3]:
print("\n--- Loading & Fixing Geometry ---")
quartiers_gdf = gpd.read_file(GEO_PATH)

# Construct Global INSEE Quartier IDs (e.g., PARIS_7511142)
quartiers_gdf['neighborhood_id'] = (
    'PARIS_751' + 
    quartiers_gdf['c_ar'].astype(str).str.zfill(2) + 
    quartiers_gdf['c_qu'].astype(str).str.zfill(2)
)

# Rename and Calculate Area
quartiers_gdf = quartiers_gdf.rename(columns={'l_qu': 'neighborhood_name'})
# Project to Lambert-93 for accurate Area calculation
quartiers_gdf['area_km2'] = quartiers_gdf.to_crs(epsg=2154).geometry.area / 10**6
quartiers_gdf = quartiers_gdf[['neighborhood_id', 'neighborhood_name', 'geometry', 'area_km2']]

VALID_QUARTIERS = set(quartiers_gdf['neighborhood_id'].unique())
print(f"✅ Loaded {len(quartiers_gdf)} neighborhoods.")

# Load IRIS Lookup Table
ref_df = pd.read_csv(REF_PATH, encoding='latin-1', dtype=str)
ref_df = ref_df[ref_df['CODE_IRIS'].str.startswith('75')]
iris_to_quartier_map = dict(zip(ref_df['CODE_IRIS'], ref_df['GRD_QUART']))


--- Loading & Fixing Geometry ---
✅ Loaded 80 neighborhoods.


## 2. Helper functions

In [4]:
def get_paris_quartier_id(iris_code, ref_map):
    """Maps IRIS code to Quartier ID."""
    code = str(iris_code).strip()
    if code in ref_map:
        return 'PARIS_' + ref_map[code]
    if code.startswith('75') and len(code) >= 7:
        potential_id = 'PARIS_' + code[:7]
        if potential_id in VALID_QUARTIERS:
            return potential_id
    return None

def process_insee_csv(filepath, year, col_mapping, ref_map, sep=None):
    """Reads CSV with explicit separator option, maps IRIS, and aggregates."""
    if not filepath.exists():
        print(f"❌ File not found: {filepath}")
        return None
        
    try:
        # Load Data
        df = pd.read_csv(filepath, sep=sep, encoding='latin-1', dtype=str, low_memory=False)
        
        # Find IRIS Column
        iris_col = next((c for c in df.columns if c.upper() in ['IRIS', 'CODE_IRIS', 'IRIS_INI']), None)
        if not iris_col:
            print(f"⚠️ {year}: No IRIS column found in {filepath.name}.")
            return None
            
        # Generate Neighborhood ID
        df['neighborhood_id'] = df[iris_col].apply(lambda x: get_paris_quartier_id(x, ref_map))
        df = df.dropna(subset=['neighborhood_id'])
        
        # Clean & Extract Data Columns
        found_any = False
        for original_col, new_name in col_mapping.items():
            actual_col = next((c for c in df.columns if c.upper() == original_col.upper()), None)
            if actual_col:
                df[new_name] = (df[actual_col].astype(str)
                               .str.replace(',', '.', regex=False)
                               .str.replace(r'\s+', '', regex=True))
                df[new_name] = pd.to_numeric(df[new_name], errors='coerce')
                found_any = True
        
        if not found_any:
            return df[['neighborhood_id']]
            
        # Aggregate
        found_metric_cols = [c for c in col_mapping.values() if c in df.columns]
        df_agg = df.groupby('neighborhood_id')[found_metric_cols].sum().reset_index()
        df_agg['year'] = year
        df_agg = df_agg[df_agg['neighborhood_id'].isin(VALID_QUARTIERS)]
        return df_agg
        
    except Exception as e:
        print(f"❌ Error processing {filepath.name}: {e}")
        return None

## 3. Processing Real Estate (DVF+)
Robust loading using DVF+ data (2014-2025) and coordinate conversion.

In [5]:
print("\n--- Processing Real Estate (DVF+) ---")
use_cols = ['anneemut', 'libnatmut', 'libtypbien', 'valeurfonc', 'sbati', 'geompar_x', 'geompar_y']
paris_sales = pd.read_csv(DVF_PATH_PLUS, usecols=use_cols, low_memory=False, sep='|')

mask = (
    (paris_sales['libnatmut'] == 'Vente') & 
    (paris_sales['libtypbien'] == 'UN APPARTEMENT') & 
    (paris_sales['valeurfonc'] > 5000) & 
    (paris_sales['sbati'] > 9) &
    (paris_sales['geompar_x'].notna())
)
paris_sales = paris_sales[mask].copy()

paris_sales['price_per_m2'] = paris_sales['valeurfonc'] / paris_sales['sbati']
paris_sales = paris_sales[paris_sales['price_per_m2'].between(1000, 40000)]
paris_sales['year'] = paris_sales['anneemut'].astype(int)

sales_gdf = gpd.GeoDataFrame(
    paris_sales, 
    geometry=gpd.points_from_xy(paris_sales.geompar_x, paris_sales.geompar_y),
    crs="EPSG:2154"
).to_crs(quartiers_gdf.crs)

sales_with_qu = gpd.sjoin(sales_gdf, quartiers_gdf[['neighborhood_id', 'geometry']], how="inner", predicate="within")

real_estate_stats = sales_with_qu.groupby(['neighborhood_id', 'year'])['price_per_m2'].agg(['median', 'std']).reset_index()
real_estate_stats = real_estate_stats.rename(columns={'median': 'median_price_per_m2', 'std': 'std_price_per_m2'})
print(f"✅ Real Estate processed.")


--- Processing Real Estate (DVF+) ---
✅ Real Estate processed.


## 4. Processing Demographics (Population, Education, Income)

In [6]:
# --- A. POPULATION ---
print("\n--- Processing Population ---")
pop_dfs = []
for year in YEARS_POP:
    if year == 2015:
        path = BASE_DIR / "POPULATION/base-ic-evol-struct-pop-2015_csv/base-ic-evol-struct-pop-2015.csv"
        cols = {'P15_POP': 'total_pop', 'P15_POP1824': 'pop_18_24', 'P15_POP2539': 'pop_25_39'} 
        sep = ',' 
    else:
        fname = f"base-ic-evol-struct-pop-{year}.CSV"
        path = BASE_DIR / "POPULATION" / f"base-ic-evol-struct-pop-{year}_csv" / fname
        yy = str(year)[-2:]
        cols = {f'P{yy}_POP': 'total_pop', f'P{yy}_POP1824': 'pop_18_24', f'P{yy}_POP2539': 'pop_25_39'}
        sep = ';'

    df = process_insee_csv(path, year, cols, iris_to_quartier_map, sep=sep)
    
    # Retry logic for 2015 typo (P0P vs POP)
    if df is not None and 'pop_18_24' not in df.columns and year == 2015:
         cols_retry = {'P15_POP': 'total_pop', 'P15_P0P1824': 'pop_18_24', 'P15_P2539': 'pop_25_39'}
         df = process_insee_csv(path, year, cols_retry, iris_to_quartier_map, sep=',')

    if df is not None and 'pop_18_24' in df.columns:
        df['young_adults_pop'] = df['pop_18_24'] + df['pop_25_39']
        pop_dfs.append(df[['neighborhood_id', 'year', 'total_pop', 'young_adults_pop']])

population_df = pd.concat(pop_dfs, ignore_index=True)

# --- B. EDUCATION (FIXED FOR 2015) ---
print("\n--- Processing Education ---")
edu_dfs = []
for year in YEARS_EDU:
    yy = str(year)[-2:]
    
    if year == 2015:
        fname = "base-ic-diplomes-formation-2015.csv"
        path = BASE_DIR / "INSTRUCTION" / fname
        sep = ','
        cols = {'P15_NSCOL15P': 'total_pop_edu', 'P15_NSCOL15P_SUP': 'higher_edu_pop'}
    else:
        fname = f"base-ic-diplomes-formation-{year}.CSV"
        path = BASE_DIR / "INSTRUCTION" / fname
        sep = ';'
        cols = {
            f'P{yy}_NSCOL15P': 'total_pop_edu',
            f'P{yy}_NSCOL15P_SUP2': 'sup2', f'P{yy}_NSCOL15P_SUP34': 'sup34', f'P{yy}_NSCOL15P_SUP5': 'sup5'
        }

    df = process_insee_csv(path, year, cols, iris_to_quartier_map, sep=sep)

    if df is not None:
        if year != 2015:
            targets = [c for c in ['sup2', 'sup34', 'sup5'] if c in df.columns]
            if targets:
                df['higher_edu_pop'] = df[targets].sum(axis=1)
            else:
                df['higher_edu_pop'] = np.nan
        
        if 'higher_edu_pop' in df.columns:
            edu_dfs.append(df[['neighborhood_id', 'year', 'total_pop_edu', 'higher_edu_pop']])

education_df = pd.concat(edu_dfs, ignore_index=True)

# --- C. INCOME (Strict Filename Targeting) ---
print("\n--- Processing Income ---")
inc_dfs = []

# Exact paths based on your provided file tree
files_map = {
    2015: {
        "path": BASE_DIR / "Income, poverty and standard of living/BASE_TD_FILO_IRIS_2015_DEC_CSV/Hoja de cálculo sin título - IRIS_DEC.csv",
        "col": "DEC_MED15",
        "sep": "," 
    },
    2019: {
        "path": BASE_DIR / "Income, poverty and standard of living/BASE_TD_FILO_IRIS_2019_DEC_CSV/BASE_TD_FILO_DEC_IRIS_2019.csv",
        "col": "DEC_MED19",
        "sep": ","
    },
    2021: {
        "path": BASE_DIR / "Income, poverty and standard of living/BASE_TD_FILO_IRIS_2021_DISP_CSV/BASE_TD_FILO_IRIS_2021_DISP.csv",
        "col": "DISP_MED21",
        "sep": ";"
    }
}

for year, config in files_map.items():
    path = config["path"]
    if not path.exists():
        print(f"❌ Missing file for {year}: {path.name}")
        continue

    print(f"   Loading {year} from {path.name}...")
    
    # Load with specific separator
    try:
        df = pd.read_csv(path, sep=config["sep"], encoding='latin-1', dtype=str, low_memory=False)
    except:
        # Fallback to python engine if C engine fails on separator
        df = pd.read_csv(path, sep=None, engine='python', encoding='latin-1', dtype=str)

    # Find IRIS Column
    iris_col = next((c for c in df.columns if 'IRIS' in c.upper()), None)
    if not iris_col:
        print(f"⚠️ {year}: No IRIS column found.")
        continue
        
    # Map IDs
    df['neighborhood_id'] = df[iris_col].apply(lambda x: get_paris_quartier_id(x, iris_to_quartier_map))
    df = df.dropna(subset=['neighborhood_id'])
    
    # Extract Median Income
    target_col = next((c for c in df.columns if c.upper() == config["col"]), None)
    if target_col:
        df['income'] = pd.to_numeric(df[target_col].str.replace(',', '.'), errors='coerce')
        
        # Aggregate to Quartier
        df_agg = df.groupby('neighborhood_id')['income'].mean().reset_index()
        df_agg.rename(columns={'income': 'median_household_income'}, inplace=True)
        df_agg['year'] = year
        df_agg = df_agg[df_agg['neighborhood_id'].isin(VALID_QUARTIERS)]
        inc_dfs.append(df_agg)
    else:
        print(f"{year}: Column {config['col']} not found.")

income_df = pd.concat(inc_dfs, ignore_index=True)


--- Processing Population ---

--- Processing Education ---

--- Processing Income ---
   Loading 2015 from Hoja de cálculo sin título - IRIS_DEC.csv...
   Loading 2019 from BASE_TD_FILO_DEC_IRIS_2019.csv...
   Loading 2021 from BASE_TD_FILO_IRIS_2021_DISP.csv...



## 5. Merging and Interpolation

In [7]:
print("\n--- Merging & Finishing ---")

# 1. Create Backbone
target_years = sorted(real_estate_stats['year'].unique())
unique_ids = quartiers_gdf['neighborhood_id'].unique()

master_df = pd.DataFrame(index=pd.MultiIndex.from_product([unique_ids, target_years], names=['neighborhood_id', 'year'])).reset_index()

# 2. Merge All Data
master_df = master_df.merge(real_estate_stats, on=['neighborhood_id', 'year'], how='left')
master_df = master_df.merge(population_df, on=['neighborhood_id', 'year'], how='left')
master_df = master_df.merge(education_df, on=['neighborhood_id', 'year'], how='left')
master_df = master_df.merge(income_df, on=['neighborhood_id', 'year'], how='left')
master_df = master_df.merge(quartiers_gdf, on='neighborhood_id', how='left')
master_df['city'] = 'Paris'

# 3. Calculate Ratios
master_df['population_density'] = master_df['total_pop'] / master_df['area_km2']
master_df['pct_young_adults'] = (master_df['young_adults_pop'] / master_df['total_pop']) * 100
master_df['pct_higher_education'] = (master_df['higher_edu_pop'] / master_df['total_pop_edu']) * 100

# 4. Spatial Interpolation & Tracking
print("Running Spatial Interpolation...")
cols_to_fill = ['median_household_income', 'pct_higher_education', 'population_density', 'pct_young_adults']

# Prepare Centroids (Projected to Metric System for Accuracy)
quartier_centroids = master_df[['neighborhood_id', 'geometry']].drop_duplicates()
quartier_centroids_gdf = gpd.GeoDataFrame(quartier_centroids, geometry='geometry')
quartier_centroids_gdf = quartier_centroids_gdf.set_crs(epsg=4326, allow_override=True).to_crs(epsg=2154)

quartier_centroids['x'] = quartier_centroids_gdf.geometry.centroid.x
quartier_centroids['y'] = quartier_centroids_gdf.geometry.centroid.y

interpolated_dfs = []

for year in master_df['year'].unique():
    year_df = master_df[master_df['year'] == year].copy()
    
    # Initialize tracking flag
    year_df['is_interpolated'] = False
    
    # Check if this year has meaningful data (ignore years like 2016 that are totally empty)
    if year_df['median_household_income'].notna().sum() > 10: 
        
        # Identify columns that are PARTIALLY missing (candidates for repair)
        current_cols_to_fill = [c for c in cols_to_fill if year_df[c].isna().sum() > 0 and year_df[c].notna().sum() > 0]
        
        if current_cols_to_fill:
            # Identify which neighborhoods are missing data BEFORE imputation
            missing_mask = year_df[current_cols_to_fill].isna().any(axis=1)
            missing_ids = year_df.loc[missing_mask, 'neighborhood_name'].tolist()
            
            print(f"   ℹ️  Year {year}: Interpolating {len(missing_ids)} neighborhoods -> {missing_ids}")
            
            # Setup KNN
            imputer = KNNImputer(n_neighbors=3, weights="distance")
            temp_df = year_df.merge(quartier_centroids[['neighborhood_id', 'x', 'y']], on='neighborhood_id')
            
            # Create Matrix [X, Y, Val1, Val2...]
            coords = temp_df[['x', 'y']].values
            values = temp_df[current_cols_to_fill].values
            
            # Fill
            filled = imputer.fit_transform(np.hstack([coords, values]))
            
            # Update DataFrame and Flag
            for i, col in enumerate(current_cols_to_fill):
                year_df[col] = filled[:, 2 + i]
            
            # Mark the neighborhoods we just fixed
            year_df.loc[missing_mask, 'is_interpolated'] = True

    interpolated_dfs.append(year_df)

master_df = pd.concat(interpolated_dfs).sort_values(['neighborhood_id', 'year'])

# 5. Homogenization (Drop rows that are still incomplete)
# This removes years like 2016-2018 where NO data existed, leaving only complete years.
clean_df = master_df.dropna(subset=cols_to_fill)

# 6. Save
final_cols = ['neighborhood_id', 'neighborhood_name', 'city', 'year', 
              'median_household_income', 'median_price_per_m2', 'std_price_per_m2', 
              'pct_higher_education', 'population_density', 'pct_young_adults', 
              'is_interpolated', 'geometry']

clean_df = clean_df[final_cols]
clean_df.to_csv(OUTPUT_DIR / "paris_master_dataframe.csv", index=False)

# save with no geometry for easier loading later
clean_df = clean_df.drop(columns=['geometry'])
clean_df.to_csv(OUTPUT_DIR / "paris_master_dataframe_no_geometry.csv", index=False)

print(f"✅ DONE. Saved clean homogeneous dataset ({len(clean_df)} rows).")
print(f"   Years kept: {sorted(clean_df['year'].unique())}")


--- Merging & Finishing ---
Running Spatial Interpolation...
   ℹ️  Year 2015: Interpolating 3 neighborhoods -> ['Champs-Elysées', "Saint-Germain-l'Auxerrois", 'Gaillon']
   ℹ️  Year 2019: Interpolating 3 neighborhoods -> ['Champs-Elysées', "Saint-Germain-l'Auxerrois", 'Gaillon']
   ℹ️  Year 2021: Interpolating 3 neighborhoods -> ['Champs-Elysées', "Saint-Germain-l'Auxerrois", 'Gaillon']
✅ DONE. Saved clean homogeneous dataset (240 rows).
   Years kept: [2015, 2019, 2021]


In [8]:
for col in final_cols:
    for year in range(2014, 2023):
        subset = master_df[master_df['year'] == year]
        n_missing = subset[col].isna().sum()
        if n_missing > 0:
            print(f"⚠️ Missing {n_missing} values for '{col}' in year {year}")
        else :
            print(f"✅ All values present for '{col}' in year {year}")


✅ All values present for 'neighborhood_id' in year 2014
✅ All values present for 'neighborhood_id' in year 2015
✅ All values present for 'neighborhood_id' in year 2016
✅ All values present for 'neighborhood_id' in year 2017
✅ All values present for 'neighborhood_id' in year 2018
✅ All values present for 'neighborhood_id' in year 2019
✅ All values present for 'neighborhood_id' in year 2020
✅ All values present for 'neighborhood_id' in year 2021
✅ All values present for 'neighborhood_id' in year 2022
✅ All values present for 'neighborhood_name' in year 2014
✅ All values present for 'neighborhood_name' in year 2015
✅ All values present for 'neighborhood_name' in year 2016
✅ All values present for 'neighborhood_name' in year 2017
✅ All values present for 'neighborhood_name' in year 2018
✅ All values present for 'neighborhood_name' in year 2019
✅ All values present for 'neighborhood_name' in year 2020
✅ All values present for 'neighborhood_name' in year 2021
✅ All values present for 'neighb